# JAX Direct-Goal Baseline

This notebook trains the final AntByte target directly from scratch: `50x50` map, `3x3` actor vision, `5` writable bits, and `10` ants. The baseline disables trainer-side pickup and distance shaping so training uses the sparse environment delivery reward only.


In [ ]:
from pathlib import Path
import os
import sys

# These must be set before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
{"project_root": PROJECT_ROOT, **runtime_status}


In [ ]:
import importlib

import jax
from tqdm.auto import tqdm

from ant_byte_env import notebook_workflows as workflows
from ant_byte_env.experiments import config_args_to_argv, load_experiment_config
from ant_byte_env.rendering import render_checkpoint
from ant_byte_env.training.jax_mappo import runner as jax_runner
from ant_byte_env.training.jax_mappo.evaluation import evaluate_checkpoint
from ant_byte_env.vault import create_vault_entry

jax_runner = importlib.reload(jax_runner)
main = jax_runner.main
ROLLOUT_TILE_SIZE = workflows.NOTEBOOK_ROLLOUT_TILE_SIZE


## Experiment Config

Edit `experiments/direct_goal_baseline.json` for durable changes to the direct-goal task or optimizer settings.


In [ ]:
EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "direct_goal_baseline.json"
experiment = load_experiment_config(EXPERIMENT_CONFIG)
if experiment.backend != "jax":
    raise ValueError(f"Expected a JAX experiment config, got {experiment.backend!r}.")

RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / "direct_goal_baseline"
MEDIA_DIR = RUN_DIR / "media"
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "model.pkl"
MEDIA_DIR.mkdir(parents=True, exist_ok=True)
ROLLOUT_POLICY_TEMPERATURE = workflows.notebook_rollout_policy_temperature(experiment.metadata)

TRAINING_ARGS = dict(experiment.args)
COMMON_ARGS = config_args_to_argv(TRAINING_ARGS)

print(f"JAX device: {jax.devices()[0]}")
print(f"Experiment config: {EXPERIMENT_CONFIG}")
print(f"Run directory: {RUN_DIR}")
COMMON_ARGS


## Quick Smoke Run

This verifies that the final target geometry compiles and trains for a tiny number of steps before launching the full baseline.


In [ ]:
smoke_args = config_args_to_argv(
    {
        **TRAINING_ARGS,
        "total_timesteps": 8,
        "num_envs": 1,
        "num_steps": 4,
        "num_minibatches": 1,
        "update_epochs": 1,
        "hidden_size": 16,
        "save_model": None,
        "load_model": None,
    }
)
smoke_metrics = main(smoke_args)
smoke_metrics


## Train Direct Baseline

This cell writes `config.json`, `metrics.jsonl`, `summary.json`, and `checkpoints/model.pkl` under `runs/notebooks/direct_goal_baseline/`.


In [ ]:
GLOBAL_UPDATE_CAP = TRAINING_ARGS["total_timesteps"] // (
    TRAINING_ARGS["num_envs"] * TRAINING_ARGS["num_steps"]
)
update_iterator = tqdm(
    range(1, GLOBAL_UPDATE_CAP + 1),
    total=GLOBAL_UPDATE_CAP,
    desc="direct goal",
    bar_format="{desc}: {n_fmt}/{total_fmt} updates |{bar}| {elapsed}<{remaining} {postfix}",
    leave=True,
)
train_history = []

def record_progress(update_index, total_updates, train_metrics):
    del total_updates
    update_iterator.update(1)
    update_iterator.set_postfix(
        loss=f"{train_metrics['loss']:.3f}",
        env=f"{train_metrics['env_return']:.3f}",
    )
    train_history.append({"update": update_index, **train_metrics})

try:
    final_train_metrics = main(
        [*COMMON_ARGS, "--run-dir", str(RUN_DIR)],
        progress_callback=record_progress,
    )
finally:
    update_iterator.close()

{
    "checkpoint": CHECKPOINT_PATH,
    "final_train_metrics": final_train_metrics,
}


## Evaluate Checkpoint

Evaluation reports raw environment delivery performance, not shaped training reward.


In [ ]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Train the direct baseline before evaluation: {CHECKPOINT_PATH}")

eval_metrics = evaluate_checkpoint(CHECKPOINT_PATH, num_episodes=8)
eval_metrics


## Optional Render and Vault

Render and archive a deterministic rollout after a checkpoint exists.


In [ ]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Train the direct baseline before rendering: {CHECKPOINT_PATH}")

rollout_path = render_checkpoint(
    CHECKPOINT_PATH,
    MEDIA_DIR / "jax_direct_goal_baseline_vision_rollout.mp4",
    backend="jax",
    tile_size=ROLLOUT_TILE_SIZE,
    policy_temperature=ROLLOUT_POLICY_TEMPERATURE,
)
vault_entry_path = create_vault_entry(
    vault_dir=RUN_DIR / "vault",
    title="JAX direct-goal baseline rollout",
    description="Vision-overlay rollout for the 50x50, 3x3-vision, 5-bit, 10-ant direct baseline.",
    assets=[rollout_path],
    metadata={
        "experiment_config": str(EXPERIMENT_CONFIG),
        "checkpoint_path": str(CHECKPOINT_PATH),
        "rollout_path": str(rollout_path),
        "eval_metrics": eval_metrics,
        "rollout_policy_temperature": ROLLOUT_POLICY_TEMPERATURE,
    },
)
{
    "rollout_path": rollout_path,
    "vault_entry_path": vault_entry_path,
}
